### Install Libraries

In [7]:
# %pip install ddgs trafilatura
%pip install openai-agents

Note: you may need to restart the kernel to use updated packages.


### Setup

In [8]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [14]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [ ]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce



#### Research Agent

In [11]:

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

#### Orchestrator Agent

In [12]:
# My original prompt
# ORCHESTRATOR_AGENT_PROMPT = """
# You are an orchestrator for multi agent researchers. Your job is to take in various research briefs and select the best one for the publication of an article.

# You MUST call for at least two completed research briefs. 

# You have access to a tool:
# - research_agent: Create a research brief based on a query

# Your typical process:
# 1. Pass your query to your research agent and obtain a research brief
# 2. Reflect on the research results — which sources look most relevant and why?
# 3. Call your research agent again for a second brief
# 4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
# 5. If you feel the research isn't enough or too similar, call your research agent again to create additional briefs.
# 6. When you have enough information, decide on the best research brief to return.

# Good research brief MUST include:
# - Key facts and statistics
# - Main themes and arguments from the sources
# - Notable data points
# - Source URLs for attribution

# Until you are ready, just keep working — reflecting as much as you need to.
# Do not rush.
# """
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs."""

orchestrator_agent = Agent(
    name="Ochestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning model
    tools=[research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistic themes, and source URLs. Pass topic as input"
    )]
)

### Let's Run it!

In [13]:
result = await Runner.run(
    orchestrator_agent,
    input = "Research the following topic and produce a comprehensive research brief: How will AI be used in Healthcare in 2030-2035",
    max_turns=30
)

  ✅ Got Results
  
 ✅ Got text: 5970 chars
  
 ✅ Got text: 9915 chars
  
 ✅ Got text: 1457 chars
  ✅ Got Results
  ❌ Failed to fetch or extract text from https://mcpress.mayoclinic.org/healthy-aging/ai-in-healthcare-the-future-of-patient-care-and-health-management/
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC12455834/
  ❌ Failed to fetch or extract text from https://www.mckinsey.com/industries/healthcare/our-insights/the-ai-powered-future-of-care
  ✅ Got Results
  
 ✅ Got text: 691 chars
  
 ✅ Got text: 6512 chars
  ✅ Got Results
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC11900311/
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/
  ✅ Got Results
  ❌ Failed to fetch or extract text from https://ep.jhu.edu/news/ai-in-healthcare-applications-and-impact/
  ❌ Failed to fetch or extra

In [15]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

Agent: Ochestrator Agent
---


Here is the selected research brief on “How AI Will Be Used in Healthcare in 2030–2035”:

Research Brief: Artificial Intelligence in Healthcare 2030–2035

Key Facts and Statistics  
• The global AI in healthcare market is projected to grow from about USD 17.2 billion in 2025 to USD 77.2 billion by 2035, at a CAGR of 16.2%. (Source: Market Research Future)  
• By 2035, AI will assist in 32% of diagnoses and 31% of therapeutic decisions, while directly impacting only about 8% of patient interactions—preserving the central role of human clinicians. (Source: HSBC Health 2035 Report)  
• Hospital-based AI applications make up 60% of the market share, with robot-assisted surgeries accounting for roughly 30%. (Source: Market Research Future)  

Main Themes and Arguments  
1. Foundational Role of AI in the Healthcare Ecosystem  
   – By 2030, AI shifts from a supplementary tool to an integral pillar in patient management, clinical decision-making, diagnostics, and drug discovery. It will drive precision medicine by fusing genomics, electronic health records, and real-time wearable data.  

2. Advanced Diagnostics and Predictive Healthcare  
   – AI-powered imaging and diagnostics will accelerate turnaround times and improve accuracy with explainable models. Predictive AI will forecast chronic diseases and readmission risks years in advance, enabling early, targeted interventions.  

3. Personalized Medicine and Drug Discovery  
   – AI-driven pharmacogenomics and genomic analytics will craft hyper-personalized oncology therapies. Generative AI will condense drug development cycles from years to weeks by rapidly identifying molecular targets.  

4. Autonomous and Augmented Surgical Robotics  
   – Semi-autonomous robotic systems, guided by AI, will enhance surgical precision, reduce complications, and expedite patient recovery by automating repeatable intra-operative tasks under human oversight.  

5. Operational Efficiency and Workforce Support  
   – Natural language processing tools will automate clinical documentation and patient triage, alleviating administrative burdens and mitigating global healthcare staffing shortages so clinicians can spend more time at the bedside.  

6. Key Technological Enablers  
   – Big data platforms with FHIR interoperability, edge computing paired with 5G/6G networks for real-time monitoring, generative AI and large language models for record-keeping and patient communication, and federated learning methods that safeguard privacy.  

7. Ethical, Regulatory, and Governance Challenges  
   – Realizing AI’s promise requires diverse, high-quality datasets to avoid bias; robust ethical and regulatory frameworks; upskilling of clinicians and policymakers; and clear assurances that AI complements rather than replaces human care to preserve patient trust.  

Notable Data Points  
• AI medical imaging systems match or exceed human performance in stroke, cancer, and cardiac diagnoses.  
• Administrative burden (45%) and physician reimbursement issues (39%) are top barriers in current healthcare, both of which AI can help alleviate.  
• Venture funding in healthcare AI startups reached USD 35 billion in H1 2024, signaling robust industry momentum.  

Source URLs  
1. https://www.eicta.iitk.ac.in/knowledge-hub/artificial-intelligence/future-of-ai-in-healthcare-predictions-innovations-2030  
2. https://www.deloitte.com/global/en/Industries/life-sciences-health-care/collections/life-sciences-and-health-care-predictions.html  
3. https://www.about.us.hsbc.com/newsroom/press-releases/artificial-intelligence-and-trust-to-reshape-healthcare-by-2035  
4. https://www.marketresearchfuture.com/reports/healthcare-artificial-intelligence-market-5681  
5. https://orthospinenews.com/2025/11/07/artificial-intelligence-in-healthcare-market-to-surge-across-usa-europe-apac-and-saudi-arabia-by-2035/

### Trace logs

https://platform.openai.com/home -> logs -> traces -> Agent workflow